### 6. Evolve individual binaries

It can be extremely useful to evolve a single binary from specific initial conditions or to start at a specific evolutionary state. This is useful for debugging binaries and checking if your custom steps or flow are working correctly.

In [2]:
import os
import shutil
import numpy as np

#### 1. Evolve a ZAMS binary

To evolve a binary from ZAMS, we will:

1. Load the standard simulation properties.
2. Load the steps.
3. Initialise the binary.
4. Evolve it.

In [2]:
from posydon.config import PATH_TO_POSYDON

path_to_params = os.path.join(PATH_TO_POSYDON, "posydon/popsyn/population_params_default.ini")
shutil.copyfile(path_to_params, './population_params_single.ini')

'./population_params_single.ini'

Here we add the metallicity to each binary system defined in the .ini file (with the function sim_kwargs['step_name'][1].update(metallicity)). It add  a new key 'metallicity' with value = 1
It will look like this : 

```python
    stop_value = None
     # float or None (only for stop_method='stop_at_condition')
    stop_interpolate = True
     # True, False
    verbose = False
     # True, False
    metallicity = 1
```

In [ ]:
# Load the function to load the simualtion properties from the ini file

from posydon.popsyn.io import simprop_kwargs_from_ini
from posydon.binary_evol.simulationproperties import SimulationProperties

# Load the simulation properties from the default ini file
sim_kwargs = simprop_kwargs_from_ini('./population_params_single.ini')

# Manually add the metallicity to each step that requires it
metallicity = {'metallicity': 1}

sim_kwargs['step_HMS_HMS'][1].update(metallicity)
sim_kwargs['step_CO_HeMS'][1].update(metallicity)
sim_kwargs['step_CO_HMS_RLO'][1].update(metallicity)
sim_kwargs['step_CO_HeMS_RLO'][1].update(metallicity)
sim_kwargs['step_detached'][1].update(metallicity)
sim_kwargs['step_disrupted'][1].update(metallicity)
sim_kwargs['step_merged'][1].update(metallicity)
sim_kwargs['step_initially_single'][1].update(metallicity)

sim_prop = SimulationProperties(**sim_kwargs)

# Load the steps and requires data
sim_prop.load_steps(verbose = True)

STEP NAME           STEP FUNCTION            KWARGS
flow (<function flow_chart at 0x14884add1c60>, {})
step_HMS_HMS (<class 'posydon.binary_evol.MESA.step_mesa.MS_MS_step'>, {'interpolation_path': None, 'interpolation_filename': None, 'interpolation_method': 'linear3c_kNN', 'save_initial_conditions': True, 'track_interpolation': False, 'stop_method': 'stop_at_max_time', 'stop_star': 'star_1', 'stop_var_name': None, 'stop_value': None, 'stop_interpolate': True, 'verbose': False, 'metallicity': 1})
step_CO_HeMS (<class 'posydon.binary_evol.MESA.step_mesa.CO_HeMS_step'>, {'interpolation_path': None, 'interpolation_filename': None, 'interpolation_method': 'linear3c_kNN', 'save_initial_conditions': True, 'track_interpolation': False, 'stop_method': 'stop_at_max_time', 'stop_star': 'star_1', 'stop_var_name': None, 'stop_value': None, 'stop_interpolate': True, 'verbose': False, 'metallicity': 1})
step_CO_HMS_RLO (<class 'posydon.binary_evol.MESA.step_mesa.CO_HMS_RLO_step'>, {'interpolation_pa

In [3]:
# Load the binary and single star class 

from posydon.binary_evol.singlestar import SingleStar
from posydon.binary_evol.binarystar import BinaryStar

In [4]:
STAR1 = SingleStar(**{'mass' : 30.782576,'state': 'H-rich_Core_H_burning'})
STAR2 = SingleStar(**{'mass' : 20.273864,'state': 'H-rich_Core_H_burning'})

BINARY = BinaryStar(STAR1, STAR2, **{'time' : 0.0, 'state' : 'detached', 'event' : 'ZAMS',
 'orbital_period' : 3513.150157, 'eccentricity' : 0.0}, properties = sim_prop)

In [5]:
# NOTE : Depending on the evolution of the binary, you might get some warnings about the Roche Lobe calculation.
BINARY.evolve()

/home/users/a/aguetar1/.conda/envs/posydon_env/lib/python3.11/site-packages/posydon/utils/common_functions.py:268: EvolutionWarning: 'Trying to compute RL radius for nonexistent object'
/home/users/a/aguetar1/.conda/envs/posydon_env/lib/python3.11/site-packages/posydon/utils/common_functions.py:283: EvolutionWarning: 'Trying to compute RL radius for nonexistent companion'
/home/users/a/aguetar1/.conda/envs/posydon_env/lib/python3.11/site-packages/posydon/utils/common_functions.py:118: ReplaceValueWarning: 'A surf_avg_omega value is negative. Setting to zero.'
/home/users/a/aguetar1/.conda/envs/posydon_env/lib/python3.11/site-packages/posydon/utils/common_functions.py:246: EvolutionWarning: 'Trying to compute RL radius for binary with invalid separation'
/home/users/a/aguetar1/.conda/envs/posydon_env/lib/python3.11/site-packages/posydon/utils/common_functions.py:263: EvolutionWarning: 'Trying to compute RL radius for nonexistent object'
/home/users/a/aguetar1/.conda/envs/posydon_env/lib

You’re now able to access the binary and evolutionary information. Using the function, <b style ='color:orange;'>BinaryStar.to_df()</b> we can inspect the evolution of the system.

In [7]:
# We will add the step names to the dataframe

col = ['time', 'step_names','event', 'orbital_period', 'eccentricity', 'S1_state', 'S2_state','S1_mass', 'S2_mass']

BINARY.to_df(extra_columns = {'step_names': 'string'})[col]

,time,step_names,event,orbital_period,eccentricity,S1_state,S2_state,S1_mass,S2_mass
binary_index,,,,,,,,,
NaN,0.000000e+00,initial_cond,ZAMS,3513.150157,0.0,H-rich_Core_H_burning,H-rich_Core_H_burning,30.782576,20.273864
NaN,6.735757e+06,step_HMS_HMS,CC1,7695.358380,0.0,stripped_He_Core_C_depleted,H-rich_Core_H_burning,13.647261,19.854313
NaN,6.735757e+06,step_SN,<NA>,NaN,NaN,BH,H-rich_Core_H_burning,13.043450,19.854313
NaN,1.016206e+07,step_disrupted,CC2,NaN,NaN,BH,H-rich_Core_C_depleted,13.043450,9.851849
NaN,1.016206e+07,step_SN,<NA>,NaN,NaN,BH,BH,13.043450,5.310433
NaN,1.016206e+07,step_end,END,NaN,NaN,BH,BH,13.043450,5.310433


##### 1.1 Re-evolving a binary

In [8]:
# restore the original state 

BINARY.restore()
BINARY.to_df()

,state,event,time,separation,orbital_period,eccentricity,rl_relative_overflow_1,rl_relative_overflow_2,lg_mtransfer_rate,mass_transfer_case,trap_radius,acc_radius,t_sync_rad_1,t_sync_conv_1,t_sync_rad_2,t_sync_conv_2,V_sys_x,V_sys_y,V_sys_z,NN_dist_x,NN_dist_y,NN_dist_z,S1_state,S1_metallicity,S1_mass,S1_log_R,S1_log_L,S1_lg_mdot,S1_lg_system_mdot,S1_lg_wind_mdot,S1_he_core_mass,S1_he_core_radius,S1_c_core_mass,S1_c_core_radius,S1_o_core_mass,S1_o_core_radius,S1_co_core_mass,S1_co_core_radius,S1_center_h1,S1_center_he4,...,S2_co_core_radius,S2_center_h1,S2_center_he4,S2_center_c12,S2_center_n14,S2_center_o16,S2_surface_h1,S2_surface_he4,S2_surface_c12,S2_surface_n14,S2_surface_o16,S2_log_LH,S2_log_LHe,S2_log_LZ,S2_log_Lnuc,S2_c12_c12,S2_center_gamma,S2_avg_c_in_c_core,S2_surf_avg_omega,S2_surf_avg_omega_div_omega_crit,S2_total_moment_of_inertia,S2_log_total_angular_momentum,S2_spin,S2_conv_env_top_mass,S2_conv_env_bot_mass,S2_conv_env_top_radius,S2_conv_env_bot_radius,S2_conv_env_turnover_time_g,S2_conv_env_turnover_time_l_b,S2_conv_env_turnover_time_l_t,S2_envelope_binding_energy,S2_mass_conv_reg_fortides,S2_thickness_conv_reg_fortides,S2_radius_conv_reg_fortides,S2_lambda_CE_1cent,S2_lambda_CE_10cent,S2_lambda_CE_30cent,S2_lambda_CE_pure_He_star_10cent,S2_total_mass_h1,S2_total_mass_he4
binary_index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
NaN,detached,ZAMS,0.0,NaN,3513.150157,0.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,H-rich_Core_H_burning,NaN,30.782576,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
BINARY.evolve()

Even though we reset the binary, the natal kick properties stored in the natal kick array are not reset. This is because we want to be able to re-evolve the binary to the same final state. If you want to reset the natal kick array values, then set its elements to <b style = 'color:red;'>None</b>.

In [11]:
# Kick mangitude km/s, azimuthal angle rad, polar angle rad and mean anomaly rad (TODO check documentation)

print('Natal kick of binary star 1 :', BINARY.star_1.natal_kick_array)
print('Natal kick of binary star 2 :',BINARY.star_2.natal_kick_array)

Natal kick of binary star 1 : [66.16708264977689, 2.108288696663849, 1.3571937596067565, 1.2859895948467197]
Natal kick of binary star 2 : [233.99228314724064, 4.634353296377042, 1.9901871388366619, 1.4592736848742986]


In [12]:
# This will be exactly the same evolution as the previous one 

BINARY.to_df(extra_columns={'step_names':'string'})[col]

,time,step_names,event,orbital_period,eccentricity,S1_state,S2_state,S1_mass,S2_mass
binary_index,,,,,,,,,
NaN,0.000000e+00,initial_cond,ZAMS,3513.150157,0.0,H-rich_Core_H_burning,H-rich_Core_H_burning,30.782576,20.273864
NaN,6.735757e+06,step_HMS_HMS,CC1,7695.358380,0.0,stripped_He_Core_C_depleted,H-rich_Core_H_burning,13.647261,19.854313
NaN,6.735757e+06,step_SN,<NA>,NaN,NaN,BH,H-rich_Core_H_burning,13.043450,19.854313
NaN,1.016206e+07,step_disrupted,CC2,NaN,NaN,BH,H-rich_Core_C_depleted,13.043450,9.851849
NaN,1.016206e+07,step_SN,<NA>,NaN,NaN,BH,BH,13.043450,5.310433
NaN,1.016206e+07,step_end,END,NaN,NaN,BH,BH,13.043450,5.310433


##### 1.2 Changing the kick

In [13]:
BINARY.restore()
BINARY.to_df()

,state,event,time,separation,orbital_period,eccentricity,rl_relative_overflow_1,rl_relative_overflow_2,lg_mtransfer_rate,mass_transfer_case,trap_radius,acc_radius,t_sync_rad_1,t_sync_conv_1,t_sync_rad_2,t_sync_conv_2,V_sys_x,V_sys_y,V_sys_z,NN_dist_x,NN_dist_y,NN_dist_z,S1_state,S1_metallicity,S1_mass,S1_log_R,S1_log_L,S1_lg_mdot,S1_lg_system_mdot,S1_lg_wind_mdot,S1_he_core_mass,S1_he_core_radius,S1_c_core_mass,S1_c_core_radius,S1_o_core_mass,S1_o_core_radius,S1_co_core_mass,S1_co_core_radius,S1_center_h1,S1_center_he4,...,S2_co_core_radius,S2_center_h1,S2_center_he4,S2_center_c12,S2_center_n14,S2_center_o16,S2_surface_h1,S2_surface_he4,S2_surface_c12,S2_surface_n14,S2_surface_o16,S2_log_LH,S2_log_LHe,S2_log_LZ,S2_log_Lnuc,S2_c12_c12,S2_center_gamma,S2_avg_c_in_c_core,S2_surf_avg_omega,S2_surf_avg_omega_div_omega_crit,S2_total_moment_of_inertia,S2_log_total_angular_momentum,S2_spin,S2_conv_env_top_mass,S2_conv_env_bot_mass,S2_conv_env_top_radius,S2_conv_env_bot_radius,S2_conv_env_turnover_time_g,S2_conv_env_turnover_time_l_b,S2_conv_env_turnover_time_l_t,S2_envelope_binding_energy,S2_mass_conv_reg_fortides,S2_thickness_conv_reg_fortides,S2_radius_conv_reg_fortides,S2_lambda_CE_1cent,S2_lambda_CE_10cent,S2_lambda_CE_30cent,S2_lambda_CE_pure_He_star_10cent,S2_total_mass_h1,S2_total_mass_he4
binary_index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
NaN,detached,ZAMS,0.0,NaN,3513.150157,0.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,H-rich_Core_H_burning,NaN,30.782576,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# [velocity, azimuthal angle, polar angle, phase]

BINARY.star_1.natal_kick_array = [1000, 0.858129274334538, 1.9157148786534735, 1.8675467897282945]
BINARY.star_2.natal_kick_array = [None, None, None, None] # Default

print('Natal kick of binary star 1 :', BINARY.star_1.natal_kick_array)
print('Natal kick of binary star 2 :', BINARY.star_2.natal_kick_array)

Natal kick of binary star 1 : [1000, 0.858129274334538, 1.9157148786534735, 1.8675467897282945]
Natal kick of binary star 2 : [None, None, None, None]


In [16]:
BINARY.evolve()
BINARY.to_df(extra_columns={'step_names':'string'})[col]

,time,step_names,event,orbital_period,eccentricity,S1_state,S2_state,S1_mass,S2_mass
binary_index,,,,,,,,,
NaN,0.000000e+00,initial_cond,ZAMS,3513.150157,0.0,H-rich_Core_H_burning,H-rich_Core_H_burning,30.782576,20.273864
NaN,6.735757e+06,step_HMS_HMS,CC1,7695.358380,0.0,stripped_He_Core_C_depleted,H-rich_Core_H_burning,13.647261,19.854313
NaN,6.735757e+06,step_SN,<NA>,NaN,NaN,BH,H-rich_Core_H_burning,13.043450,19.854313
NaN,1.016206e+07,step_disrupted,CC2,NaN,NaN,BH,H-rich_Core_C_depleted,13.043450,9.851849
NaN,1.016206e+07,step_SN,<NA>,NaN,NaN,BH,BH,13.043450,5.310433
NaN,1.016206e+07,step_end,END,NaN,NaN,BH,BH,13.043450,5.310433


##### 1.3 Evolve from a specific step

In [ ]:
BINARY.restore(3) # Restore to step 3
BINARY.to_df()

#### 2. Loading a binary from an existing population 

For this one, we will use <b style='color:orange;'> BBH_contact.h5 </b> file and <b style='color:red;'>ITS</b> corresponding file, i.e. <b style='color:orange;'>population_params2.ini</b>. 

In [18]:
from posydon.popsyn.synthetic_population import Population

pop = Population('BBH_contact.h5')

Missing ini parameter: orbital_separation_scheme
Missing ini parameter: orbital_separation_min
Missing ini parameter: orbital_separation_max


In [20]:
# Because the original population contains two extra columns, these have to be defined manually here.
# Otherwise, resetting the binaries will fail. 

BINARY = BinaryStar.from_df(pop.history[0], extra_columns = {'step_names':'string', 'step_times':'float'})

# You have to set the simulation properties for the binary 
BINARY.propertires = sim_prop

# Set the natal kick arrays from the onelne to get the exat same evolution as the original binary
BINARY.star_1.natal_kick_array = pop.oneline[0][['S1_natal_kick_array_0', 'S1_natal_kick_array_1', 
                                                 'S1_natal_kick_array_2', 'S1_natal_kick_array_3']].values.flatten()
BINARY.star_2.natal_kick_array = pop.oneline[0][['S2_natal_kick_array_0', 'S2_natal_kick_array_1', 
                                                 'S2_natal_kick_array_2', 'S2_natal_kick_array_3']].values.flatten()

In [21]:
BINARY.to_df(extra_columns={'step_names':'string'})[col]

,time,step_names,event,orbital_period,eccentricity,S1_state,S2_state,S1_mass,S2_mass
binary_index,,,,,,,,,
0,0.000000e+00,initial_cond,ZAMS,2.601094e+00,0.000000,H-rich_Core_H_burning,H-rich_Core_H_burning,39.680646,28.818876
0,5.884773e+06,step_HMS_HMS,CC1,2.870397e+00,0.000000,stripped_He_Core_C_depleted,H-rich_Core_H_burning,17.887336,33.766741
0,5.884773e+06,step_SN,<NA>,3.369315e+00,0.099667,BH,H-rich_Core_H_burning,17.448928,33.766741
0,6.377155e+06,step_detached,oRLO2,3.274130e+00,0.000000,BH,H-rich_Core_H_burning,17.448928,34.848539
0,7.785821e+06,step_CO_HMS_RLO,CC2,3.399253e+00,0.000000,BH,stripped_He_Core_C_depleted,18.029109,16.336831
0,7.785821e+06,step_SN,<NA>,3.125831e+00,0.101857,BH,BH,18.029109,16.033050
0,1.061960e+10,step_dco,CO_contact,3.452800e-08,0.000000,BH,BH,18.029109,16.033050
0,1.061960e+10,step_end,END,3.452800e-08,0.000000,BH,BH,18.029109,16.033050


In [22]:
BINARY.restore()

In [24]:
BINARY.evolve()
BINARY.to_df(extra_columns={'step_names':'string'})[col]

AttributeError: 'tuple' object has no attribute 'get'

#### 3. Evolving a Binary star starting from an arbitrary state

If you want to evolve a custom binary star, you can do so by manually creating the <I style='color:red;'>BinaryStar</I> object and providing the <I style='color:red;'>SimulationProperties</I> object. 

Here is an example of a neutron star with a low-mass helium companion in Roche Lobe overflow. 

In [25]:
# We initialise the binary to be tight and Roche lobe overflowing in a circular orbit.
binary = BinaryStar(star_1 = SingleStar(**{'state' : 'NS',
                                           'mass' : 1.1,
                                           'spin' : 0.,}),
                    star_2 = SingleStar(**{'state' : 'H-rich_Core_H_burning',
                                           'mass' : 2.5,
                                           'natal_kick_array' : [10., 0., 0., 0.]}),
                    **{'time' : 0.,
                       'state' : 'RLO2',
                       'event' : 'oRLO2',
                       'orbital_period' : 1.,
                       'eccentricity' : 0.},
                    properties = sim_prop,
                    )
binary.evolve()
binary.to_df(extra_columns={'step_names':'string'})[col]

,time,step_names,event,orbital_period,eccentricity,S1_state,S2_state,S1_mass,S2_mass
binary_index,,,,,,,,,
NaN,0.000000e+00,initial_cond,oRLO2,1.000000,0.0,NS,H-rich_Core_H_burning,1.100000,2.500000
NaN,2.352552e+08,step_CO_HMS_RLO,oCE2,0.085162,0.0,NS,H-rich_Core_H_burning,1.893069,0.164589
NaN,2.352552e+08,step_CE,oMerging2,0.085162,0.0,NS,H-rich_Core_H_burning,1.893069,0.164589
NaN,2.352552e+08,step_merged,<NA>,NaN,NaN,NS,massless_remnant,1.893069,0.000000
NaN,2.352552e+08,step_end,END,NaN,NaN,NS,massless_remnant,1.893069,0.000000


In [26]:
# We perform the same evolution, but with a stripped helium star to go to the CO_HeMS_RLO grid
binary = BinaryStar(star_1 = SingleStar(**{'state' : 'NS',
                                           'mass' : 1.1,
                                           'spin' : 0.,}),
                    star_2 = SingleStar(**{'state' : 'stripped_He_Core_He_burning',
                                           'mass' : 2.5,
                                           'natal_kick_array' : [10., 0., 0., 0.]}),
                    **{'time' : 0.,
                       'state' : 'RLO2',
                       'event' : 'oRLO2',
                       'orbital_period' : 1.,
                       'eccentricity' : 0.},
                    properties = sim_prop,
                    )
binary.evolve()
binary.to_df(extra_columns={'step_names':'string'})[col]

/home/users/a/aguetar1/.conda/envs/posydon_env/lib/python3.11/site-packages/posydon/interpolation/IF_interpolation.py:1173: InterpolationWarning: '1NN interpolation used for 1 binaries out of hull. Parameter-wise distance (Unnormalized) for point with maximum out-of-hull euclidian distance (Normalized): [0.08012213 2.38677534 2.23387895]'


,time,step_names,event,orbital_period,eccentricity,S1_state,S2_state,S1_mass,S2_mass
binary_index,,,,,,,,,
NaN,0.000000e+00,initial_cond,oRLO2,1.000000,0.000000,NS,stripped_He_Core_He_burning,1.100000,2.500000
NaN,2.271486e+04,step_CO_HeMS_RLO,CC2,0.630731,0.000000,NS,stripped_He_Core_C_depleted,1.100794,1.748416
NaN,2.271486e+04,step_SN,<NA>,1.100199,0.266997,NS,NS,1.100794,1.277584
NaN,1.380000e+10,step_dco,maxtime,0.931619,0.231711,NS,NS,1.100794,1.277584
NaN,1.380000e+10,step_end,END,0.931619,0.231711,NS,NS,1.100794,1.277584


We will change the binary to a much wider system and have it start in the detached state. This requires additional starting information to allow the star to be matched to the single star grids.

In [27]:
from posydon.utils.constants import Zsun
from posydon.utils.common_functions import orbital_separation_from_period

import numpy as np
Z = 1.0

# Setup the central abundances
zams_table = {2.: 2.915e-01,
                      1.: 2.703e-01,
                      0.45: 2.586e-01,
                      0.2: 2.533e-01,
                      0.1: 2.511e-01,
                      0.01: 2.492e-01,
                      0.001: 2.49e-01,
                      0.0001: 2.49e-01}

Y = zams_table[Z]
Z = Z*Zsun
X = 1 - Y - Z

In [28]:
STAR1 = SingleStar(**{'mass':1.2,
                      'state': 'NS'})

STAR2 = SingleStar(**{'mass': 17.782576,
                      'state': 'H-rich_Core_H_burning',
                      # add the metallicity and central abundances
                      'metallicity':Z,
                      'center_h1':X,
                      'center_he4':Y,
                      # add a numerical value for the radius
                      'log_R': np.nan,
                      # add the helium core mass
                      'he_core_mass': 0.0,
                      })

binary = BinaryStar(STAR1, STAR2,
                    **{'time' : 0.,
                       'state' : 'detached',
                       'event' : None,
                       'orbital_period' : 5000.,
                        # calculate the separation; current bug that the separation is not automatically calculated if orbital period is given
                       'separation': orbital_separation_from_period(5000., 17.782576, 1.2),
                       'eccentricity' : 0.},
                    properties = sim_prop,
                    )
binary.evolve()
binary.to_df(extra_columns={'step_names':'string'})[col]

/home/users/a/aguetar1/.conda/envs/posydon_env/lib/python3.11/site-packages/posydon/interpolation/IF_interpolation.py:1475: InterpolationWarning: 'normalization was skipped during interpolation: c=initial_MT, inds=[0]'
/home/users/a/aguetar1/.conda/envs/posydon_env/lib/python3.11/site-packages/posydon/interpolation/IF_interpolation.py:1499: InterpolationWarning: 'de-normalization was skipped during interpolation: c=initial_MT, inds=[0]'


,time,step_names,event,orbital_period,eccentricity,S1_state,S2_state,S1_mass,S2_mass
binary_index,,,,,,,,,
NaN,0.000000e+00,initial_cond,<NA>,5000.000000,0.0,NS,H-rich_Core_H_burning,1.20000,17.782576
NaN,1.116364e+07,step_detached,oRLO2,2504.644491,0.0,NS,H-rich_Core_He_burning,1.20000,15.839361
NaN,1.116458e+07,step_CO_HMS_RLO,oCE2,1983.523944,0.0,NS,H-rich_Core_He_burning,1.20001,15.533734
NaN,1.116458e+07,step_CE,<NA>,0.038323,0.0,NS,stripped_He_Core_He_burning,1.20001,6.405666
NaN,NaN,step_CO_HeMS,<NA>,NaN,0.0,NS,stripped_He_Core_He_burning,NaN,NaN
NaN,NaN,step_end,END,NaN,0.0,NS,stripped_He_Core_He_burning,NaN,NaN
